# Competitor client extraction

Reads an Excel export of resumes, works out which competitor each **Company** value refers to, and pulls out any client named alongside it (for example `Shore360- Infinite Energy` → competitor *Shore360*, client *Infinite Energy*).

**How to run:** fill in Step 1, then use **Run → Run All Cells** (or Kernel → Restart & Run All). Nothing else needs editing for a normal run.

**What you get:** one Excel workbook (`OUTPUT_FILE`) with these sheets:
- **All Rows**: your original data, untouched, with extracted columns added on the right
- **Needs Review**: the rows a person should check, one line per distinct Company text, biggest first
- **Client Summary**: the competitor → client map with row counts
- **Client Implied**: rows with no client named but a non-BPO industry, which suggests a client exists
- **Manual Label Check**: rows where your own Client flag / Possible Client label differs from the extraction (only if those columns exist)
- **Read Me**: how to fill in the review columns

Plus two files you fill in and keep (you or an LLM), both rewritten each run with your answers kept and a dated backup made first:
- **`client_terms_review.xlsx`** (Step 9): every extracted client name, most common first, with possible misspellings side by side. Mark each KEEP, IGNORE (not a client, e.g. *Manila*, *APAC*) or type the corrected name; the next run applies it everywhere.
- **`accuracy_check.xlsx`** (Step 10): a random sample of results to check by hand or by LLM, scored every run so you can see the accuracy. Names already decided in the terms file are left out of the sample.

**Files:** everything is read from and written to the `DATA_DIR` folder set in Step 1.

**Review loop:** fill in the yellow columns on the Needs Review sheet, save it under a new name (e.g. `client_extraction_reviewed.xlsx`) in the same folder, point `REVIEWED_FILE` at it, and run again. Your decisions are applied to every row with that exact Company text, now and on future exports.

## Step 1: Settings

In [ ]:
from pathlib import Path

# Folder the notebook reads from and writes to. Keep the r before the quotes: it stops Python
# from treating the backslashes in a Windows path as special characters.
DATA_DIR = Path(r"C:\Users\Alfonso(FonsoP)Panli\Desktop\Projects\Competitor Analysis (client scrape)\data")

INPUT_FILE = DATA_DIR / "candidate_tracker_report_extract.xlsx"   # your export (.xlsx, or .csv)
SHEET_NAME = 0                          # sheet name, or 0 for the first sheet
COMPANY_COLUMN = "Company"              # column holding the employer text

# Optional columns: set to None if your file doesn't have them
ID_COLUMN = "JXID"                      # candidate ID, used to count distinct people
INDUSTRY_COLUMN = "Industry"            # used for the Client Implied sheet
MANUAL_FLAG_COLUMN = "Client flag"      # your hand labels, used to check the extraction
MANUAL_CLIENT_COLUMN = "Possible Client"

OUTPUT_FILE = DATA_DIR / "client_extraction_output.xlsx"   # rewritten on every run

# A previous output file whose Needs Review sheet you have filled in. None on the first run.
# Example: REVIEWED_FILE = DATA_DIR / "client_extraction_reviewed.xlsx"
REVIEWED_FILE = None

# Files you fill in and keep under the same name. Each run reads your answers from them, then
# rewrites them with fresh counts, keeping every answer. The old version is copied to
# DATA_DIR\backups first. Close them in Excel before running.
TERMS_FILE = DATA_DIR / "client_terms_review.xlsx"   # Step 9: is each extracted client name a real client?
CHECK_FILE = DATA_DIR / "accuracy_check.xlsx"        # Step 10: random sample to measure accuracy

# Step 10: how many unchecked lines to keep ready in CHECK_FILE, per extraction status
CHECK_SAMPLE = {"client found": 40, "no client": 10}
CHECK_SEED = 7                          # change it to draw a different random sample

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {DATA_DIR}")
if not Path(INPUT_FILE).exists():
    found = sorted(p.name for p in DATA_DIR.glob("*.xls*")) + sorted(p.name for p in DATA_DIR.glob("*.csv"))
    raise FileNotFoundError(f"No file called '{Path(INPUT_FILE).name}' in {DATA_DIR}. Files there: {found}")
_files = {k: Path(v).resolve() for k, v in dict(INPUT_FILE=INPUT_FILE, OUTPUT_FILE=OUTPUT_FILE,
          REVIEWED_FILE=REVIEWED_FILE, TERMS_FILE=TERMS_FILE, CHECK_FILE=CHECK_FILE).items() if v}
if len(set(_files.values())) < len(_files):
    raise ValueError(f"Two settings point at the same file; each needs its own name: {_files}")
print(f"Reading  {INPUT_FILE}\nWriting  {OUTPUT_FILE}")

# Industries that just mean "worked at a BPO" and don't hint at a client
BPO_INDUSTRIES = ["Call Center / IT-Enabled Services / BPO"]

## Step 2: Matching rules

You only need to touch this cell when the Needs Review sheet shows a gap.

- **`COMPETITORS`**: each competitor and the regex patterns for its spellings. If you see `no competitor matched` for a real competitor spelling, add a pattern here. Brands and legal entities that belong to a competitor (MyCloud People for TOA, ShoreAgent for Shore360) go here too, so they aren't mistaken for clients.
- **`SUFFIX_WORDS`**: words that can trail a competitor name and belong to the competitor, not to a client (Inc, Philippines, Pampanga, WFH…).
- **`REVIEW_PATTERNS`**: known traps. Any Company text matching one of these is sent to review with the note shown.
- **`SIMILAR_SHOW` / `SIMILAR_IGNORE`**: how alike two names must be to be shown as possible misspellings, or to be ignored along with a name you marked IGNORE.
- **`SUSPECT_WORDS` / `LINKING_WORDS`**: only used for hints in the terms file. Nothing is removed because of them.

In [ ]:
import re
COMPETITORS = {
    "TOA Global": [r"the\s+outsourced\s+accou\w*s?(\s+global)?", r"\btoa\b(\s+global)?", r"my\s*cloud\s+people"],
    "Shore360": [r"shore\s*360", r"shore\s*(agent|suite|remote)"],
    "Emapta": [r"emapta(\s+global)?"],
    "Beepo": [r"beepo"],
    "SupportNinja": [r"support\s*ninja"],
    "Boomering": [r"boomering"],
    "ConnectOS": [r"connect\s*os\b"],
    "Connext": [r"connext"],
    "MicroSourcing": [r"micro\s*sourcing"],
    "Remote Staff": [r"remote\s+staff"],
    "Remote Employee": [r"remote\s+employee"],
    "KMC Solutions": [r"\bkmc\b(\s+(mag|savills))?"],
    "Booth & Partners": [r"booth\s*(&|and)\s*partners"],
    "Cyberbacker": [r"cyberbacker"],
    "Hammerjack": [r"hammer\s*jack"],
    "Staff Domain": [r"staff\s*domain"],
    "Penbrothers": [r"pen\s*brothers"],
    "Peak Support": [r"peak\s+support"],
    "Sourcefit": [r"sourcefit"],
    "MyOutDesk": [r"my\s*out\s*desk"],
    "Wing Assistant": [r"wing\s+assistant"],
    "Magellan Solutions": [r"magellan\s+solutions"],
    "Infinit-O": [r"infinit-?o(utsourcing)?\b"],
    "DOXA Talent": [r"doxa\s*(talent|7)?"],
    # Not seen in the sample yet; patterns ready for the full file
    "Outsourced.ph": [r"outsourced\.?\s*ph\b"],
    "Cloudstaff": [r"cloud\s*staff\b"],
    "Support Services Group": [r"support\s+services\s+group", r"\bssg\b"],
    "Engaged Solutions": [r"engaged\s+solutions"],
    "Cloud Accountant Staffing": [r"cloud\s+accountant\s+staffing"],
    "Flatworld Solutions": [r"flat\s*world\s+solutions"],
    "Auxis": [r"\bauxis\b"],
    "Virtual Latinos": [r"virtual\s+latinos"],
    "Athena BPO S.A.S": [r"athena\s+bpo"],
    "Athena": [r"\bathena\b"],
    "Unity Communications": [r"unity\s+communications"],
    "SixEleven Global": [r"six\s*eleven"],
    "Hatchit Solutions": [r"hatchit"],
    "Hire Horatio": [r"hire\s+horatio", r"\bhoratio\b"],
    "Somewhere": [r"\bsomewhere\b"],
    "Pearl Talent": [r"pearl\s+talent"],
    "Helpware": [r"helpware"],
}

SUFFIX_WORDS = (r"inc|incorporated|imc|corp|corporation|llc|pty|ltd|limited|ph|phils?|philippines|bpo|"
                r"global|enterprises?|enterpise|international|versatile|services|solutions?|outsourcing|"
                r"remote|staffing|agency|pampanga|cebu|clark|clarkfield|angeles|city|freeport|zone|wfh|"
                r"eastwood|libis|quezon|makati|taguig|bgc|ortigas|pasig|davao|iloilo|phil|branch|core")

# Phrases describing the job or the competitor rather than a client; removed before parsing
DESCRIPTORS = [r"business\s+process\s+outsource\s+company", r"australian\s+outsourcing\s+company",
               r"outsource\s+company", r"bpo\s+company", r"contract/temporary", r"\bkpo\b",
               r"seat\s+leasing", r"\breal\s+estate\b(?!\s+software)"]

# Client market hints when no client is named
REGION_PATTERNS = {r"\bus\s+and\s+au\s+clients?\b": "US+AU", r"\bau\s+clients?\b": "AU",
                   r"\bus\s+clients?\b": "US", r"\bnew\s+zealand\s+clients?\b": "NZ",
                   r"\buk\s+clients?\b": "UK", r"australian\s+(account|firm)": "AU"}

# Known traps: (pattern on the raw Company text, note shown to the reviewer)
REVIEW_PATTERNS = [
    (r"\btoa\s+corporation\b", "TOA Corporation is a different company (Japanese construction)"),
    (r"^\s*remote\s+employee(\s+bpo)?(\s+ph|\s+phils?)?\s*$", "'Remote Employee' may describe the job, not the competitor"),
    (r"^\s*(athena|somewhere)\s*$", "generic word: may not be the competitor"),
    (r"doxa\s*7", "DOXA7 may not be DOXA Talent"),
    (r"salmat", "may be an acquired/predecessor entity, not a client"),
    (r"outsource\s+desk|jobs360", "may be a competitor brand/entity, not a client"),
    (r";", "semicolon list: probably several employers, not clients"),
]


# ---- Client terms file (Step 9) ----
SIMILAR_SHOW = 0.80     # names at least this alike (0 to 1) are listed side by side as possible misspellings
SIMILAR_IGNORE = 0.88   # a name this close to one you marked IGNORE is ignored too (names of 5+ letters).
                        # The terms file says when this happened; mark that name KEEP to stop it.

# Words that are rarely a client name on their own. Only used for hints in the terms file:
# nothing is removed because of this list. Removing is done with IGNORE in the terms file.
SUSPECT_WORDS = {
    "manila", "makati", "cebu", "pasig", "taguig", "bgc", "ortigas", "clark", "pampanga", "davao", "iloilo",
    "philippines", "asia", "apac", "pacific", "emea", "global", "international", "intl", "regional",
    "headquarters", "hq", "office", "site", "branch", "hub", "center", "centre", "campus",
    "group", "community", "careers", "career", "seeker", "team", "teams", "department", "division",
    "service", "services", "solution", "solutions", "s", "e-services", "teleservices", "operations",
    "offshore", "staffing", "outsourcing", "outsource", "outsourced", "bpo", "kpo", "agency", "va",
    "virtual", "assistant", "assistants", "support", "account", "accounts", "accountant", "accountants",
    "accounting", "healthcare", "financial", "finance", "seasonal", "project", "projects", "client",
    "clients", "inc", "incs", "pte", "pty", "ltd", "corp", "company", "and", "&", "the",
}
LINKING_WORDS = {"via", "in", "at", "under", "c/o", "and", "&", "for", "of", "with", "from", "to", "by"}

## Step 3: Extraction logic (no need to edit)

In [ ]:
from collections import Counter, defaultdict
from difflib import SequenceMatcher

SUFFIX = rf"(?:[\s,.\-–—()|]*\b(?:{SUFFIX_WORDS})\b\.?)*"
COMPILED = {c: [re.compile(rf"(?:\bthe\s+)?{p}{SUFFIX}", re.I) for p in ps] for c, ps in COMPETITORS.items()}
CONNECTORS = r"^(?:client|clients|under|through|at|for|via|the)\b[\s:]*|[\s:]*\b(?:client|clients|under|through|at)$"
LEGAL_TAIL = r"[\s,.]*\b(pty\.?\s*ltd|pty|ltd|limited|inc|corp|llc)\b\.?$"
GEO = r"(australia|au|us|usa|uk|nz|new zealand|nsw|vic|victoria|queensland|qld|virginia|united states of america|united states|canada|philippines|ph)"
JUNK = {"agency", "bpo", "philippines", "ph", "client", "clients", "cpas", "cpa", "inc"}
SPLIT = r"\s*(?:[/|;()–—,]|\s-\s|\s-|-\s|\bclient:)\s*"

def parse(raw):
    """Company text -> competitor, client candidates, client locations, region hint, review flags."""
    s = re.sub(r"\s+", " ", str(raw)).strip()
    flags = [note for pat, note in REVIEW_PATTERNS if re.search(pat, s, re.I)]
    region = ""
    for p, r in REGION_PATTERNS.items():
        if re.search(p, s, re.I):
            region = r; s = re.sub(p, " ", s, flags=re.I)
    for d in DESCRIPTORS:
        s = re.sub(d, " ", s, flags=re.I)

    competitor, rest = "", s
    for comp, rxs in COMPILED.items():
        for rx in rxs:
            if rx.search(rest):
                competitor = competitor or comp
                rest = rx.sub(" | ", rest)
    if not competitor:
        return dict(competitor="", clients=[], locations=[], region=region,
                    flags=flags + ["no competitor matched: new spelling? add it in Step 2"])

    clients, locations, former = [], [], []
    for seg in re.split(SPLIT, rest, flags=re.I):
        seg, prev = seg.strip(" .,-:'\""), None
        while prev != seg:                    # peel connector words, legal tails, place names
            prev = seg
            if seg.lower().startswith("formerly "):
                former.append(seg[9:]); seg = ""
            seg = re.sub(CONNECTORS, "", seg, flags=re.I).strip(" .,-:")
            seg = re.sub(LEGAL_TAIL, "", seg, flags=re.I).strip(" .,-:")
            m = re.search(rf"\s\(?{GEO}\)?$", seg, re.I)
            if m and len(seg) > len(m.group(0)) + 2:
                locations.append(m.group(1)); seg = seg[:m.start()].strip(" .,-:")
        if not seg or len(re.sub(r"\W", "", seg)) < 3 or seg.lower() in JUNK:
            continue
        if re.fullmatch(GEO, seg, re.I):
            locations.append(seg); continue
        clients.append(seg)

    if former:
        flags.append("former client name: " + "; ".join(former))
    # "several candidates" is flagged in Step 6, after names marked IGNORE in the terms file are removed
    return dict(competitor=competitor, clients=clients, locations=locations, region=region, flags=flags)

def name_key(n):
    k = re.sub(r"[^a-z0-9]", "", str(n).lower())
    return re.sub(r"(company|co)$", "", k) or k     # a name that is just "Company" keeps its key

def canonicalise(names, preferred=()):
    """Group spelling variants of the same client. Reviewer-typed names win as the display name."""
    groups = []
    for n in list(preferred) + list(names):
        k = name_key(n)
        if not k:
            continue
        for g in groups:
            close = SequenceMatcher(None, k, g["key"]).ratio() > 0.9
            prefix = min(len(k), len(g["key"])) >= 6 and (k.startswith(g["key"]) or g["key"].startswith(k))
            if close or prefix:
                g["names"][n] += 1; break
        else:
            groups.append({"key": k, "names": Counter({n: 1})})
    canon = {}
    for g in groups:
        pref = [n for n in g["names"] if n in preferred]
        best = pref[0] if pref else g["names"].most_common(1)[0][0]
        letters = [ch for ch in best if ch.isalpha()]
        if not pref and " " in best and letters and sum(ch.isupper() for ch in letters) > 0.8 * len(letters):
            best = best.title()
        for n in g["names"]:
            canon[n] = best
    return canon

def same_client(a, b):
    ka, kb = name_key(a), name_key(b)
    return bool(ka and kb) and (ka in kb or kb in ka or SequenceMatcher(None, ka, kb).ratio() > 0.85)


# ---- Helpers for the terms file (Step 9), the accuracy check (Step 10) and writing Excel ----
import shutil
from datetime import datetime
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

def close_matches(key, keys, cutoff):
    """Name keys from `keys` at least `cutoff` alike (0 to 1) to `key`, closest first."""
    sm, found = SequenceMatcher(), []
    sm.set_seq2(key)
    for k in keys:
        if k == key or 2 * min(len(k), len(key)) / (len(k) + len(key)) < cutoff:
            continue
        sm.set_seq1(k)
        if sm.real_quick_ratio() >= cutoff and sm.quick_ratio() >= cutoff:
            r = sm.ratio()
            if r >= cutoff:
                found.append((k, r))
    return sorted(found, key=lambda x: -x[1])

def parse_term_decision(text):
    """Decision cell in the terms file -> (action, new name). action: ignore / keep / rename."""
    t = str(text).strip()
    if t.upper() in ("IGNORE", "NONE"):
        return "ignore", ""
    if t.upper() in ("KEEP", "OK"):
        return "keep", ""
    return "rename", t

def term_hints(name, competitors):
    """Reasons a client name may not be a real client. Hints only: they never change the result."""
    words = [w for w in re.split(r"[^a-z0-9&/\-]+", name.lower()) if w]
    hints = []
    if words and all(w in SUSPECT_WORDS for w in words):
        hints.append("only common words")
    if words and (words[0] in LINKING_WORDS or words[-1] in LINKING_WORDS):
        hints.append("starts or ends with a linking word: may need a corrected name")
    if len(name_key(name)) <= 3:
        hints.append("very short")
    comp_words = {w for c in competitors for w in re.split(r"[^a-z0-9]+", c.lower()) if len(w) >= 4}
    if comp_words & {w for w in words if len(w) >= 4}:
        hints.append("shares a word with the competitor's name")
    n_comp = len([c for c in competitors if c])
    if n_comp >= 3:
        hints.append(f"seen under {n_comp} competitors")
    return hints

HEADER_FILL = PatternFill("solid", start_color="1F3864")
ADDED_FILL = PatternFill("solid", start_color="2E75B6")
INPUT_FILL = PatternFill("solid", start_color="FFFF00")
BODY_FONT = Font(name="Arial", size=10)          # created once and shared: much faster on big sheets
TOP, WRAP_TOP = Alignment(vertical="top"), Alignment(vertical="top", wrap_text=True)

def write_book(path, sheets, inputs=(), highlight=None, dropdowns=None, backup=False):
    """Write {sheet name: table} to a formatted workbook.
    inputs:    columns a reviewer fills in (yellow)
    highlight: {sheet: [columns]} given a lighter blue header (columns added by the notebook)
    dropdowns: {column: [choices]} offered as a dropdown; other typed values are still allowed
    backup:    copy the existing file to DATA_DIR/backups before overwriting it"""
    path, highlight, dropdowns = Path(path), highlight or {}, dropdowns or {}
    if backup and path.exists():
        bdir = path.parent / "backups"
        bdir.mkdir(exist_ok=True)
        shutil.copy2(path, bdir / f"{path.stem}_{datetime.now():%Y%m%d_%H%M%S}{path.suffix}")
    try:
        with pd.ExcelWriter(path, engine="openpyxl") as xw:
            for name, frame in sheets.items():
                frame.to_excel(xw, sheet_name=name, index=False)
            for ws in xw.book.worksheets:
                name, frame = ws.title, sheets[ws.title]
                align = WRAP_TOP if name == "Read Me" else TOP
                for row in ws.iter_rows():
                    for cell in row:
                        cell.font = BODY_FONT
                        if cell.row > 1:
                            cell.alignment = align
                for j, col in enumerate(frame.columns, start=1):
                    h = ws.cell(row=1, column=j)
                    h.font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
                    h.fill = ADDED_FILL if col in highlight.get(name, ()) else HEADER_FILL
                    letter = get_column_letter(j)
                    if col in inputs:
                        h.fill, h.font = INPUT_FILL, Font(name="Arial", size=10, bold=True)
                        for r in range(2, ws.max_row + 1):
                            ws.cell(row=r, column=j).fill = INPUT_FILL
                    if col in dropdowns and ws.max_row >= 2:
                        dv = DataValidation(type="list", formula1='"' + ",".join(dropdowns[col]) + '"',
                                            allow_blank=True, showErrorMessage=False)
                        ws.add_data_validation(dv)
                        dv.add(f"{letter}2:{letter}{ws.max_row}")
                    width = max([len(str(col))] + [len(str(v)) for v in frame[col].head(500)]) if len(frame) else len(col)
                    ws.column_dimensions[letter].width = min(max(width + 2, 10), 70)
                if name == "Read Me":
                    ws.column_dimensions["A"].width = 140
                else:
                    ws.freeze_panes = "A2"
                    ws.auto_filter.ref = ws.dimensions
            if "Read Me" in sheets:
                xw.book.move_sheet("Read Me", offset=-(len(sheets) - 1))
                xw.book.active = 1
    except PermissionError:
        raise PermissionError(f"Can't write {path.name}: close it in Excel, then run this cell again.") from None

## Step 4: Load the data

In [ ]:
import pandas as pd

if Path(INPUT_FILE).suffix.lower() == ".csv":
    df = pd.read_csv(INPUT_FILE, dtype=str)
else:
    df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, dtype=str)
df = df.fillna("")

if COMPANY_COLUMN not in df.columns:
    raise KeyError(f"No column called '{COMPANY_COLUMN}'. Columns found: {list(df.columns)}")
for name in ["ID_COLUMN", "INDUSTRY_COLUMN", "MANUAL_FLAG_COLUMN", "MANUAL_CLIENT_COLUMN"]:
    col = globals()[name]
    if col and col not in df.columns:
        print(f"Note: optional column '{col}' not found, skipping it.")
        globals()[name] = None

print(f"{len(df):,} rows, {df[COMPANY_COLUMN].str.strip().replace('', pd.NA).nunique():,} distinct Company values")

## Step 5: Load earlier review decisions (if any)

In [ ]:
decisions = {}
if REVIEWED_FILE:
    rev = pd.read_excel(REVIEWED_FILE, sheet_name="Needs Review", dtype=str).fillna("")
    for _, r in rev.iterrows():
        comp, client = r["Reviewer: Competitor"].strip(), r["Reviewer: Client"].strip()
        if comp or client:
            decisions[r["Company"].strip()] = (comp, client)
    print(f"Loaded {len(decisions):,} review decisions from {REVIEWED_FILE}")
else:
    print("No review file set; running on automatic extraction only.")

# Client name decisions from the terms file (Step 9): IGNORE, KEEP or a corrected name
term_rows, term_decisions = pd.DataFrame(), {}
if Path(TERMS_FILE).exists():
    term_rows = pd.read_excel(TERMS_FILE, sheet_name="Client Terms", dtype=str).fillna("")
    for _, r in term_rows.iterrows():
        key = name_key(r["Client"])
        if key and r["Decision"].strip():
            action, new = parse_term_decision(r["Decision"])
            term_decisions[key] = dict(action=action, new=new, client=r["Client"].strip())
    print(f"Loaded {len(term_decisions):,} client name decisions from {Path(TERMS_FILE).name}:",
          dict(Counter(d["action"] for d in term_decisions.values())))
else:
    print(f"No {Path(TERMS_FILE).name} yet: Step 9 will create it.")

## Step 6: Run the extraction

In [ ]:
def resolve(raw):
    """Automatic result for one Company text, with any reviewer decision applied on top."""
    p = parse(raw)
    p["source"] = "auto"
    if raw in decisions:
        comp, client = decisions[raw]
        if comp.upper() == "NONE":
            p.update(competitor="", clients=[])
        elif comp and comp.upper() != "OK":
            p["competitor"] = comp
        if client.upper() == "NONE":
            p["clients"] = []
        elif client and client.upper() != "OK":
            p["clients"] = [c.strip() for c in client.split(";") if c.strip()]
        p.update(flags=[], source="reviewer")
    return p

companies = df[COMPANY_COLUMN].astype(str).str.strip()
parsed = {c: resolve(c) for c in companies.unique() if c}
reviewer_names = {c for p in parsed.values() if p["source"] == "reviewer" for c in p["clients"]}
canon = canonicalise([c for p in parsed.values() for c in p["clients"]], preferred=sorted(reviewer_names))

# Apply the client name decisions from the terms file. They match on the name after spelling
# variants are grouped, so one decision covers every spelling in the group.
spellings = defaultdict(set)                     # grouped name -> spellings seen
for p in parsed.values():
    for c in p["clients"]:
        spellings[canon.get(c, c)].add(c)
ignore_keys = [k for k, d in term_decisions.items() if d["action"] == "ignore" and len(k) >= 5]

_term_cache = {}
def term_action(name):
    """Grouped client name -> (action, new name, what happened). action: ignore / keep / rename / ''."""
    if name not in _term_cache:
        keys = [name_key(name)] + sorted(name_key(s) for s in spellings.get(name, ()))
        hit = next((term_decisions[k] for k in keys if k in term_decisions), None)
        if hit:
            how = {"ignore": "ignored", "keep": "kept", "rename": f"renamed to '{hit['new']}'"}[hit["action"]]
            res = (hit["action"], hit["new"], how)
        else:
            close = close_matches(keys[0], ignore_keys, SIMILAR_IGNORE) if len(keys[0]) >= 5 else []
            res = (("ignore", "", f"ignored: close spelling of ignored '{term_decisions[close[0][0]]['client']}'")
                   if close else ("", "", ""))
        _term_cache[name] = res
    return _term_cache[name]

for raw, p in parsed.items():
    names = list(dict.fromkeys(canon.get(c, c) for c in p["clients"]))
    p["terms"], p["term_decided"] = names, False      # names before term decisions: Step 9 counts these
    if p["source"] == "reviewer":
        p["clients"] = names
        continue
    final = []
    for n in names:
        action, new, _ = term_action(n)
        p["term_decided"] |= bool(action)
        if action != "ignore":
            final.append(new if action == "rename" else n)
    p["clients"] = list(dict.fromkeys(final))
    if len(p["clients"]) > 1 and ";" not in raw:
        p["flags"].append("several candidates: check which is the client")

rows = []
for i, raw in companies.items():
    p = parsed.get(raw, dict(competitor="", clients=[], locations=[], region="", flags=[], source="auto"))
    clients = list(p["clients"])
    flags = list(p["flags"])

    manual = ""
    if MANUAL_FLAG_COLUMN and str(df.at[i, MANUAL_FLAG_COLUMN]).strip() not in ("", "0"):
        label = df.at[i, MANUAL_CLIENT_COLUMN] if MANUAL_CLIENT_COLUMN else ""
        manual = "agrees" if any(same_client(c, label) for c in clients) else "DISAGREES"

    implied = ""
    if INDUSTRY_COLUMN and not clients and p["competitor"]:
        ind = str(df.at[i, INDUSTRY_COLUMN]).strip()
        if ind and ind not in BPO_INDUSTRIES:
            implied = ind

    if not raw:
        status = "blank"
    elif p["source"] == "reviewer":
        status = "reviewed"
    elif flags:
        status = "needs review"
    elif clients:
        status = "client found"
    elif p["region"]:
        status = "region only"
    else:
        status = "no client"

    rows.append({"Competitor": p["competitor"], "Extracted Client": "; ".join(clients),
                 "Client Location": "; ".join(dict.fromkeys(p["locations"])), "Region Hint": p["region"],
                 "Extraction Status": status, "Review Notes": " | ".join(flags),
                 "Client Implied by Industry": implied, "Matches Manual Label": manual})

out = pd.concat([df, pd.DataFrame(rows, index=df.index)], axis=1)
out["Extraction Status"].value_counts()

## Step 7: Build the review list and the summaries

In [ ]:
# Needs Review: one line per distinct Company text, most frequent first
nr = out[out["Extraction Status"] == "needs review"]
review = (nr.groupby(COMPANY_COLUMN)
            .agg(Rows=(COMPANY_COLUMN, "size"), Competitor=("Competitor", "first"),
                 **{"Extracted Client": ("Extracted Client", "first"),
                    "Why Flagged": ("Review Notes", lambda s: " | ".join(dict.fromkeys(" | ".join(s).split(" | "))))})
            .reset_index().rename(columns={COMPANY_COLUMN: "Company"})
            .sort_values(["Rows", "Company"], ascending=[False, True]))
for col in ["Reviewer: Competitor", "Reviewer: Client", "Reviewer: Notes"]:
    review[col] = ""

# Client Summary: competitor -> client with confirmed and pending row counts
pairs = defaultdict(lambda: {"Confirmed Rows": 0, "Rows Pending Review": 0, "Locations": Counter()})
for _, r in out[out["Extracted Client"] != ""].iterrows():
    for c in r["Extracted Client"].split("; "):
        d = pairs[(r["Competitor"], c)]
        d["Rows Pending Review" if r["Extraction Status"] == "needs review" else "Confirmed Rows"] += 1
        for loc in filter(None, r["Client Location"].split("; ")):
            d["Locations"][loc] += 1
summary = pd.DataFrame([{"Competitor": k[0], "Client": k[1], "Confirmed Rows": v["Confirmed Rows"],
                         "Rows Pending Review": v["Rows Pending Review"], "Client Locations": ", ".join(v["Locations"])}
                        for k, v in pairs.items()])
if len(summary):
    summary = summary.sort_values(["Competitor", "Confirmed Rows", "Client"], ascending=[True, False, True])

# Client Implied: no client named, but the industry suggests one
implied = (out[out["Client Implied by Industry"] != ""]
           .groupby(["Competitor", "Client Implied by Industry"]).size().reset_index(name="Rows")
           .rename(columns={"Client Implied by Industry": "Industry"})
           .sort_values("Rows", ascending=False))

# Manual Label Check: rows where your hand label and the extraction differ.
# Kept separate from Needs Review because your label may come from the resume body, which
# the Company text can't show; a decision here shouldn't apply to every row with the same text.
manual_check = pd.DataFrame()
if MANUAL_FLAG_COLUMN:
    keep = [COMPANY_COLUMN] + [c for c in [MANUAL_CLIENT_COLUMN, INDUSTRY_COLUMN] if c] + ["Competitor", "Extracted Client"]
    manual_check = out.loc[out["Matches Manual Label"] == "DISAGREES", keep].copy()
    manual_check.insert(0, "Row in All Rows", manual_check.index + 2)

print(f"Needs Review: {len(review):,} distinct Company values covering {review['Rows'].sum():,} rows")
print(f"Client Summary: {len(summary):,} competitor-client pairs")
if MANUAL_FLAG_COLUMN:
    print("Check against your manual labels:", out["Matches Manual Label"].replace("", pd.NA).value_counts().to_dict())
review.head(15)

## Step 8: Write the Excel file

In [ ]:
readme = pd.DataFrame({"How to review": [
    "1. Go to the Needs Review sheet. Each line is one distinct Company text; Rows shows how many resumes use it, so start at the top.",
    "2. Read the Company text and Why Flagged, then fill in the yellow columns:",
    "   Reviewer: Competitor  →  OK (extraction is right), NONE (not a competitor at all), or type the correct competitor. Blank = accept.",
    "   Reviewer: Client      →  OK (extraction is right), NONE (no client here), or type the correct client. Separate several with ;",
    "   Example: Company 'TOA Global; Deloitte Philippines; Scrubbed.Net'  →  Competitor: OK, Client: NONE, Notes: list of employers",
    "3. Leave both yellow columns blank to skip a line; it will come back next run.",
    "4. Save a copy under a new name in the data folder (e.g. client_extraction_reviewed.xlsx), set REVIEWED_FILE to it in Step 1, and Run All.",
    "   Decisions apply to every row with that exact Company text. Use a new name so the next run doesn't overwrite your review.",
    "5. When a line says 'no competitor matched' and it IS a competitor, also add the spelling in Step 2 so future exports catch it automatically.",
    "",
    "Names that are never clients (Manila, APAC, Careers…) are handled in client_terms_review.xlsx, not here: one IGNORE there removes the name everywhere.",
    "Manual Label Check lists rows where your own label differs from the extraction. Fix those in your source data, not here.",
    "Extraction Status on All Rows: client found · region only (market known, client unnamed) · no client · needs review · reviewed · blank",
]})

sheets = {"All Rows": out, "Needs Review": review, "Client Summary": summary, "Client Implied": implied}
if MANUAL_FLAG_COLUMN:
    sheets["Manual Label Check"] = manual_check
sheets["Read Me"] = readme
write_book(OUTPUT_FILE, sheets, inputs=[c for c in review.columns if c.startswith("Reviewer:")],
           highlight={"All Rows": list(rows[0].keys()) if rows else []})
print(f"Saved {OUTPUT_FILE}")

## Step 9: Client terms file

Writes `TERMS_FILE`: one line per extracted client name, most common first, with hints and similar spellings. Your KEEP / IGNORE / corrected-name decisions in it were already applied in Step 6; this step refreshes the counts and keeps every answer.

In [ ]:
# One line per client name the extraction found, most common first. Counts are taken BEFORE your
# decisions are applied, so an ignored name keeps showing how many rows it removes.
TERM_INPUTS = ["Decision", "Decided By", "Notes"]
TERM_COLUMNS = ["Client", "Rows", "Candidates", "Hints"] + TERM_INPUTS + [
    "This Run", "Similar Names", "Competitors", "Spellings Seen", "Example Company Text"]

text_rows = companies[companies != ""].value_counts()
text_ids = df.groupby(companies)[ID_COLUMN].agg(set) if ID_COLUMN else pd.Series(dtype=object)

stats = defaultdict(lambda: {"rows": 0, "ids": set(), "comps": Counter(), "examples": Counter()})
for raw, p in parsed.items():
    if p["source"] == "reviewer":                 # already decided line by line in Needs Review
        continue
    n = int(text_rows.get(raw, 0))
    for name in p["terms"]:
        s = stats[name]
        s["rows"] += n
        s["ids"] |= text_ids.get(raw, set())
        s["comps"][p["competitor"]] += n
        s["examples"][raw] += n

by_key = defaultdict(list)
for name in stats:
    by_key[name_key(name)].append(name)
all_keys = list(by_key)

# Answers already in the terms file, matched on the name or on any spelling in its group
prev_answers = {}
for _, r in term_rows.iterrows():
    a = {c: str(r.get(c, "")).strip() for c in TERM_INPUTS}
    if any(a.values()) and name_key(r["Client"]):
        prev_answers[name_key(r["Client"])] = (r["Client"], a)

lines, used = [], set()
for name, s in stats.items():
    key = name_key(name)
    _, _, how = term_action(name)
    similar = []
    if len(key) >= 4:
        for k2, _ in close_matches(key, all_keys, SIMILAR_SHOW)[:5]:
            similar += [f"{n2} ({stats[n2]['rows']})" for n2 in by_key[k2]]
    hints = term_hints(name, list(s["comps"]))
    if how.startswith("ignored: close"):
        hints.insert(0, how[len("ignored: "):])
    answers = dict.fromkeys(TERM_INPUTS, "")
    for k in [key] + sorted(name_key(sp) for sp in spellings.get(name, ())):
        if k in prev_answers and k not in used:
            answers = prev_answers[k][1]
            used.add(k)
            break
    lines.append({"Client": name, "Rows": s["rows"], "Candidates": len(s["ids"]) if ID_COLUMN else "",
                  "Hints": "; ".join(hints), **answers, "This Run": how or "no decision",
                  "Similar Names": "; ".join(similar),
                  "Competitors": "; ".join(f"{c} ({n})" for c, n in s["comps"].most_common(4)),
                  "Spellings Seen": "; ".join(sorted(sp for sp in spellings.get(name, ()) if sp != name)),
                  "Example Company Text": " || ".join(t for t, _ in s["examples"].most_common(3))})

# Keep answers for names that don't appear in this export, so no decision is ever lost
for k, (client, a) in prev_answers.items():
    if k not in used:
        lines.append({"Client": client, "Rows": 0, "Candidates": 0 if ID_COLUMN else "", "Hints": "", **a,
                      "This Run": "not in current data", "Similar Names": "", "Competitors": "",
                      "Spellings Seen": "", "Example Company Text": ""})

terms = (pd.DataFrame(lines, columns=TERM_COLUMNS)
           .sort_values(["Rows", "Client"], ascending=[False, True]))

terms_readme = pd.DataFrame({"How to use this file": [
    "Every client name the extraction found, one line each, most common first. Rows and Candidates are counted",
    "before your decisions, so an ignored name still shows how many rows it removes.",
    "",
    "Fill in the yellow columns (you or an LLM), save this file under the SAME name, and run the notebook again:",
    "   Decision    →  KEEP (a real client), IGNORE (not a client: remove it wherever it appears), or type the corrected",
    "                  name, e.g. 'Proposition Marketing' for 'Proposition Marketing via'. Blank = not decided yet.",
    "                  A decision covers every spelling in Spellings Seen.",
    "   Decided By  →  Manual or LLM, so you can tell later who made each decision.",
    "   Notes       →  anything you want to remember.",
    "",
    "Misspellings: Similar Names lists names that look alike, with their row counts, so you can decide them together.",
    "A name very close to one you marked IGNORE (e.g. 'Manilla' after 'Manila') is ignored automatically; This Run and",
    "Hints say so. Mark it KEEP if that was wrong.",
    "",
    "This Run shows what the last run did with each name: ignored, kept, renamed, or no decision.",
    "Hints are only clues (only common words, starts/ends with a linking word, very short, seen under many competitors);",
    "they never change anything by themselves.",
    "",
    "Each run rewrites this file with fresh counts and keeps every answer. The previous version is copied to the",
    "backups folder first. Close the file in Excel before running.",
]})

write_book(TERMS_FILE, {"Client Terms": terms, "Read Me": terms_readme}, inputs=TERM_INPUTS,
           dropdowns={"Decision": ["IGNORE", "KEEP"], "Decided By": ["Manual", "LLM"]}, backup=True)
acted = terms["This Run"].str.split(":").str[0].value_counts().to_dict()
print(f"Saved {TERMS_FILE}: {len(terms):,} client names, {(terms['Decision'] == '').sum():,} without a decision")
print("This run:", acted)
print(f"Undecided names cover {terms.loc[terms['Decision'] == '', 'Rows'].sum():,} rows")
terms.head(15)

## Step 10: Accuracy check file

Writes `CHECK_FILE`: a random sample of results to check (by you or an LLM), skipping Company texts whose names are already decided in the terms file. Every checked line is re-scored against this run, and the Score sheet shows the accuracy.

In [ ]:
# A random sample of extraction results for you (or an LLM) to check, to measure accuracy.
# Company texts whose client names already have a decision in the terms file (Step 9) are left out.
CHECK_INPUTS = ["Correct Competitor", "Correct Client", "Checked By", "Notes"]
CHECK_COLUMNS = (["Company", "Rows", "Status When Added", "Competitor", "Extracted Client", "Added On"]
                 + CHECK_INPUTS + ["Current Competitor", "Current Client", "Result"])

check = (pd.read_excel(CHECK_FILE, sheet_name="Check", dtype=str).fillna("") if Path(CHECK_FILE).exists()
         else pd.DataFrame(columns=CHECK_COLUMNS))
check = check.reindex(columns=CHECK_COLUMNS, fill_value="")
check["Company"] = check["Company"].str.strip()

current = {raw: (p["competitor"], p["clients"]) for raw, p in parsed.items()}
status_of = out.assign(_text=companies).drop_duplicates("_text").set_index("_text")["Extraction Status"]

def answered(r):
    return bool(r["Correct Competitor"].strip() or r["Correct Client"].strip())

def correct_answer(r):
    """The checker's answer as (competitor, clients). OK or blank = the extraction shown on that line was right."""
    cc, cl = r["Correct Competitor"].strip(), r["Correct Client"].strip()
    comp = r["Competitor"] if cc.upper() in ("", "OK") else ("" if cc.upper() == "NONE" else cc)
    if cl.upper() == "NONE" or (cc.upper() == "NONE" and not cl):
        clients = []
    elif cl.upper() in ("", "OK"):
        clients = [c for c in r["Extracted Client"].split("; ") if c]
    else:
        clients = [c.strip() for c in cl.split(";") if c.strip()]
    return comp, clients

def same_name(a, b):
    return (not a and not b) or same_client(a, b)

def result(r):
    if not answered(r):
        return "not checked yet"
    if r["Company"] not in current:
        return "not in current data"
    comp, clients = correct_answer(r)
    now_comp, now_clients = current[r["Company"]]
    ok_comp = same_name(comp, now_comp)
    ok_client = (len(clients) == len(now_clients)
                 and all(any(same_name(a, b) for b in now_clients) for a in clients)
                 and all(any(same_name(a, b) for b in clients) for a in now_clients))
    if ok_comp and ok_client:
        return "correct"
    return "wrong client" if ok_comp else "wrong competitor" if ok_client else "wrong competitor and client"

# Top up: keep CHECK_SAMPLE unchecked lines ready per status
is_open = ~check.apply(answered, axis=1) & check["Company"].isin(current) if len(check) else pd.Series(dtype=bool)
new_lines = []
for status, target in CHECK_SAMPLE.items():
    need = target - int((check.loc[is_open, "Status When Added"] == status).sum()) if len(check) else target
    pool = sorted(t for t, p in parsed.items()
                  if status_of.get(t) == status and t not in set(check["Company"]) and not p["term_decided"])
    if need <= 0 or not pool:
        continue
    for t in pd.Series(pool).sample(min(need, len(pool)), random_state=CHECK_SEED + len(check)):
        comp, clients = current[t]
        new_lines.append({"Company": t, "Status When Added": status, "Competitor": comp,
                          "Extracted Client": "; ".join(clients), "Added On": f"{datetime.now():%Y-%m-%d}"})
if new_lines:
    check = pd.concat([check, pd.DataFrame(new_lines)], ignore_index=True).reindex(columns=CHECK_COLUMNS).fillna("")

# Refresh against this run's extraction, so a rule change shows up in Result straight away
check["Rows"] = check["Company"].map(text_rows).fillna(0).astype(int)
check["Current Competitor"] = check["Company"].map(lambda t: current.get(t, ("", []))[0])
check["Current Client"] = check["Company"].map(lambda t: "; ".join(current.get(t, ("", []))[1]))
check["Result"] = check.apply(result, axis=1) if len(check) else ""
check["_open"] = check["Result"] == "not checked yet"
check = check.sort_values(["_open", "Added On", "Rows"], ascending=[False, True, False]).drop(columns="_open")

# Score: share of checked lines the current extraction gets right
done = check[check["Result"].str.startswith(("correct", "wrong"))]
def score_row(group, g):
    ok, w = g["Result"] == "correct", g["Rows"]
    return {"Group": group, "Lines Checked": len(g), "Correct": int(ok.sum()),
            "Accuracy": f"{ok.mean():.0%}" if len(g) else "",
            "Accuracy Weighted by Rows": f"{(w * ok).sum() / w.sum():.0%}" if w.sum() else ""}
score = pd.DataFrame([score_row("All checked lines", done)]
                     + [score_row(f"Status when added: {s}", g) for s, g in done.groupby("Status When Added")]
                     + [score_row(f"Checked by: {b or '(blank)'}", g) for b, g in done.groupby("Checked By")])
mistakes = done["Result"].value_counts().drop("correct", errors="ignore")
score = pd.concat([score, pd.DataFrame({"Group": "Mistake type: " + mistakes.index.astype(str),
                                        "Lines Checked": mistakes.values})], ignore_index=True).fillna("")

check_readme = pd.DataFrame({"How to use this file": [
    "A random sample of extraction results, to measure how often the automatic extraction is right.",
    "Company texts whose client names you already decided in client_terms_review.xlsx are not sampled.",
    "",
    "Fill in the yellow columns (you or an LLM), save this file under the SAME name, and run the notebook again:",
    "   Correct Competitor  →  OK (Competitor is right), NONE (not a competitor), or type the right one. Blank = OK.",
    "   Correct Client      →  OK (Extracted Client is right), NONE (no client named), or type the right client(s), separated by ;",
    "   Checked By          →  Manual or LLM, so the Score sheet can compare the two.",
    "A line counts as checked once either Correct column is filled in.",
    "",
    "Each run re-scores every checked line against the latest extraction: Current Competitor / Current Client / Result.",
    "So after you change a rule or a terms decision you can see straight away whether it fixed (or broke) a checked line.",
    "Answers here only measure. They don't change the extraction: fix a wrong result in the terms file, Needs Review or Step 2.",
    "",
    "Status When Added: 'client found' lines measure how often a found client is real; 'no client' lines measure missed clients.",
    "Each run tops up the sample so there are always new unchecked lines (numbers set by CHECK_SAMPLE in Step 1).",
    "The previous version is copied to the backups folder before each run. Close the file in Excel before running.",
]})

write_book(CHECK_FILE, {"Check": check, "Score": score, "Read Me": check_readme}, inputs=CHECK_INPUTS,
           highlight={"Check": ["Current Competitor", "Current Client", "Result"]},
           dropdowns={"Correct Competitor": ["OK", "NONE"], "Correct Client": ["OK", "NONE"],
                      "Checked By": ["Manual", "LLM"]}, backup=True)
print(f"Saved {CHECK_FILE}: {len(check):,} lines, {len(new_lines)} added this run, {len(done)} checked")
score